In [2]:
import sys, os
sys.path.insert(0, os.path.abspath("src"))
import re
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import importlib, core
importlib.reload(core)
from core import (count_sites_per_sample_ptm_report, process_ptm_site_report, _hex_to_rgba)

In [3]:
# Shared palettes (match Figure 3 v01)
color_sequence_red    = ['#FBA08D', '#FA7A61', '#F95534', '#ED2E07', '#CB2706', '#9E1E05']
color_sequence_violet = ['#C79EEA', '#B178E2', '#9B52DA', '#7E2AC7', '#6D25AD', '#551D87']
HELA_CELL_ORDER = [100, 300, 500, 1000, 2000, 3000]
STEM_CELL_ORDER = [100, 300, 500, 800, 1000, 3000]
PROC_META = {'Protein_group', 'Gene_group', 'PTM_0_aa', 'PTM_pos', 'PTM_mult123',
             'PTM_flank', 'PTM_Collapse_key', 'PTM_localization', 'UPD_seq'}

# Data upload

In [4]:
# Revision Supplementary Figure 2 data = wide PTM Site Reports (per-run localization
# at 0.75), same format/source as Figure 3 (raw_data/revision/figure3). Bucketed by
# workflow x system; the stem-cell SORTED files are multi-condition (cellline1..5).
RAW_DIR = Path('raw_data/revision/figure3')

def parse_filename(fname):
    workflow = 'nanophos' if 'nanoPhos' in fname else ('uphos' if 'uPhos' in fname else 'unknown')
    if 'HeLa' in fname:
        system = 'hela'
    elif 'StemCells' in fname:
        system = 'stem'
    else:
        system = 'unknown'
    m = re.search(r'(\d+)cells', fname)
    return workflow, system, (int(m.group(1)) if m else None)

buckets = {}
for path in sorted(RAW_DIR.glob('*.tsv')):
    if 'RAlin' in path.name or 'normalized' in path.name:
        continue 
    w, s, n = parse_filename(path.name)
    if n is None:
        continue
    buckets.setdefault(f'{w}_{s}', {})[n] = path

loaded = {}
for group, files in buckets.items():
    loaded[group] = {n: pd.read_csv(files[n], sep='\t', low_memory=False) for n in sorted(files)}
    print(f'{group:<16} cells: {sorted(files)}')

hela_nanophos = loaded.get('nanophos_hela', {})
hela_uphos    = loaded.get('uphos_hela',    {})
stem_sorted   = loaded.get('nanophos_stem', {})

uphos_hela       cells: [100, 300, 500, 1000, 2000, 3000]
nanophos_hela    cells: [100, 300, 500, 1000, 2000, 3000]
nanophos_stem    cells: [100, 300, 500, 800, 1000, 3000]


# Supplementary Figure 2a
Coefficient of variation of Class I phosphosite intensities across the 3 HeLa nanoPhos replicates, per sorted-cell number. Per project policy, CV is a per-feature quantitative metric — multiplicity is KEPT (no collapse).

In [5]:
# Suppl 2a - CV of Class I phosphosite intensities (HeLa nanoPhos), per cell number.
# Strict per-run Class I via process_ptm_site_report; CV computed in LINEAR space across
# the 3 replicates, requiring all 3 valid (matches Figure3_v00 logic). Multiplicity KEPT.
cv_by_cells = {}
for n in HELA_CELL_ORDER:
    if n not in hela_nanophos:
        continue
    sd = process_ptm_site_report(hela_nanophos[n], cutoff=0.75)['site_data']
    cols = [c for c in sd.columns if c not in PROC_META]
    lin = np.power(2.0, sd[cols])                       # log2 -> linear
    n_valid = lin.notna().sum(axis=1)
    cv = lin.std(axis=1) / lin.mean(axis=1)             # ddof=1 (pandas default), as in v00
    cv = cv.where(n_valid >= len(cols))                 # require all replicates valid
    cv_by_cells[n] = cv.dropna()

all_cv = np.concatenate([v.values for v in cv_by_cells.values()])
median_cv = float(np.nanmedian(all_cv))
print('median CV (all cell numbers): %.3f  (%.1f%%)' % (median_cv, 100 * median_cv))
for n, v in cv_by_cells.items():
    print(f'  {n:>4} cells: n={len(v):>5}  median CV={np.nanmedian(v):.3f}')

order = [n for n in HELA_CELL_ORDER if n in cv_by_cells]
fig = go.Figure()
for i, n in enumerate(order):
    fig.add_trace(go.Box(
        y=cv_by_cells[n], x=[str(n)] * len(cv_by_cells[n]), name=str(n),
        marker_color=color_sequence_red[i], line=dict(color=color_sequence_red[i]),
        boxpoints='outliers',                           # show outlier points beyond whiskers
        marker=dict(size=4, opacity=0.5), showlegend=False,
    ))
fig.add_hline(y=median_cv, line={'dash': 'dash', 'width': 2, 'color': 'black'})
fig.update_layout(width=600, height=600, template='plotly_white',
                  xaxis_title='Sorted cells', yaxis_title='Coefficient of variation')
fig.update_xaxes(categoryorder='array', categoryarray=[str(n) for n in order])
fig.update_yaxes(rangemode='tozero')
fig.show()
fig.write_image(r'D:\Projects\nanoPhos\figures_upd\figure3\suppl_figure2a.pdf', width=600, height=600)

Dropped 2,625 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 8,040 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 4,189 → 4,171.
Final: 4,171 sites × 3 samples.
Dropped 3,318 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 19,590 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 10,100 → 10,064.
Final: 10,064 sites × 3 samples.
Dropped 4,147 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 26,100 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 13,235 → 13,177.
Final: 13,177 sites × 3 samples.
Dropped 4,890 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Pe

# Supplementary Figure 2b
Phosphopeptide selectivity (% phospho precursors) per sorted-cell number. Selectivity is a precursor-level enrichment-specificity metric, independent of Class I site localization — the original per-cell-number selectivity exports are reused unchanged.

In [6]:
# Suppl 2b - selectivity (REUSED original exports; not affected by Class I recount).
# Source: per-cell-number selectivity tables from the original figure3 analysis.
SEL_DIR = Path(r'Z:\Denys_nanoPhos\PRIDE\analysis_data\figure3')
sel_cell_order = [100, 300, 500, 1000, 2000, 3000]

sel = []
for n in sel_cell_order:
    df = pd.read_csv(SEL_DIR / f'selectivity_{n}cells.tsv', sep='\t')
    sel.append(df.set_index('XLabel').apply(np.sum, axis=1).tolist())

ids = np.repeat([str(n) for n in sel_cell_order], [len(s) for s in sel]).tolist()
df_sel = pd.DataFrame({'Selectivity': [v for s in sel for v in s], 'ID': ids})
print(df_sel.groupby('ID', sort=False)['Selectivity'].agg(['count', 'mean']).round(2).to_string())

fig = px.strip(df_sel, y='Selectivity', x='ID', orientation='h',
               category_orders={'ID': [str(n) for n in sel_cell_order]})
fig.update_layout(width=600, height=600, template='plotly_white', showlegend=False,
                  xaxis_title='Sorted cells', yaxis_title='Selectivity (%)')
fig.update_traces(marker=dict(size=18, color='#db4c2e', line=dict(width=0.5, color='black')))
fig.update_yaxes(range=[0, 100], showgrid=True, gridwidth=0.1, gridcolor='#F3F2F2')
fig.show()
# fig.write_image(r'D:\Projects\nanoPhos\figures_upd\figure3\suppl_figure2b.pdf', width=600, height=600)

      count   mean
ID                
100       3  65.41
300       3  85.71
500       3  88.80
1000      3  88.18
2000      3  89.68
3000      3  88.96


# Supplementary Figure 2c
uPhos Class I phosphosite depth vs sorted-cell number (HeLa). Counterpart to Figure 3a (nanoPhos). Class I, multiplicity collapsed, localization enforced.

In [7]:
# Suppl 2c - uPhos HeLa Class I phosphosite depth vs cell number (box + points, n=3).
# Same hardened counter and styling as Figure 3a, applied to the uPhos arm.
counts_by_cells = {n: list(count_sites_per_sample_ptm_report(hela_uphos[n]).values())
                   for n in HELA_CELL_ORDER if n in hela_uphos}

rows = []
for n in HELA_CELL_ORDER:
    if n not in counts_by_cells:
        continue
    c = counts_by_cells[n]
    rows.append({'cells': n, 'n_reps': len(c), 'median': int(np.median(c)),
                 'mean': int(np.mean(c)),
                 'CV%': round(100 * np.std(c, ddof=1) / np.mean(c), 1) if len(c) > 1 else 0})
print(pd.DataFrame(rows).set_index('cells').to_string())

BOX_COLOR, POINT_COLOR = '#7E2AC7', '#393E46'     # violet box to distinguish from nanoPhos (3a)
order = [n for n in HELA_CELL_ORDER if n in counts_by_cells]
fig = go.Figure()
for n in order:
    ys = counts_by_cells[n]
    fig.add_trace(go.Box(
        y=ys, x=[str(n)] * len(ys), name=str(n),
        boxpoints='all', jitter=0.3, pointpos=0,
        marker=dict(size=9, color=POINT_COLOR, line=dict(width=0.5, color='black')),
        line=dict(color=BOX_COLOR, width=1.5), fillcolor=_hex_to_rgba(BOX_COLOR, 0.15),
        showlegend=False,
    ))
fig.update_layout(template='plotly_white', width=600, height=600, showlegend=False,
                  xaxis_title='Sorted cells', yaxis_title='Class I phosphosites (uPhos)')
fig.update_xaxes(categoryorder='array', categoryarray=[str(n) for n in order])
fig.update_yaxes(range = [0, 7100])
fig.show()
fig.write_image(r'D:\Projects\nanoPhos\figures_upd\figure3\suppl_figure2c.pdf', width=600, height=600)

       n_reps  median  mean   CV%
cells                            
100         3     569   501  24.6
300         3    2087  2073  11.0
500         3    3031  3178  11.5
1000        3    3936  3955   1.4
2000        3    5880  5865   1.0
3000        3    5844  6001   4.9


# Supplementary Figure 2d
Per-state Class I phosphosite depth across sorted-cell numbers for the stem-cell sorted series. States 2iL/SL/RA (cellline 1/2/4), consistent with the rest of Figure 3 (cellline3 and cellline5=RA24 excluded). One separate plot per cell number.

In [8]:
# Suppl 2d - stem-cell Class I depth per state (2iL/SL/RA); ONE SEPARATE PLOT per cell number.
# 500 cells omitted here (shown in the main figure). Class I, multiplicity collapsed,
# localization enforced. cellline1=2iL, 2=SL, 4=RA.
CELLLINE_TO_STATE = {1: '2iL', 2: 'SL', 4: 'RA'}     # cellline3 & 5(RA24) excluded
STATE_ORDER = ['2iL', 'SL', 'RA']
STATE_COLOR = {'2iL': '#ED2E07', 'SL': '#7E2AC7', 'RA': '#2A88C7'}
STEM_PANELS = [n for n in STEM_CELL_ORDER if n != 500]   # 500 is in the main figure

def state_of(col):
    m = re.search(r'cellline(\d+)', col)
    return CELLLINE_TO_STATE.get(int(m.group(1))) if m else None

depth_by_cells = {}
for n in STEM_PANELS:
    if n not in stem_sorted:
        continue
    counts = count_sites_per_sample_ptm_report(stem_sorted[n])
    by_state = {s: [] for s in STATE_ORDER}
    for sample, c in counts.items():
        st = state_of(sample)
        if st in by_state:
            by_state[st].append(c)
    depth_by_cells[n] = by_state
    line = '  '.join(f'{s}: mean={int(np.mean(by_state[s]))}' for s in STATE_ORDER if by_state[s])
    print(f'{n:>4} cells | {line}')

# one independent figure per cell number; y-axis 0 -> 1.1 x (max point in that panel)
for n in [c for c in STEM_PANELS if c in depth_by_cells]:
    fig = go.Figure()
    panel_max = max(v for vals in depth_by_cells[n].values() for v in vals)
    for s in STATE_ORDER:
        ys = depth_by_cells[n][s]
        if not ys:
            continue
        fig.add_trace(go.Box(
            y=ys, x=[s] * len(ys), name=s, boxpoints='all', jitter=0.3, pointpos=0,
            marker=dict(size=9, color=STATE_COLOR[s], line=dict(width=0.5, color='black')),
            line=dict(color=STATE_COLOR[s], width=1.5),
            fillcolor=_hex_to_rgba(STATE_COLOR[s], 0.2), showlegend=False))
    fig.update_layout(width=500, height=500, template='plotly_white', showlegend=False,
                      title=f'{n} cells',
                      xaxis_title='Pluripotency state', yaxis_title='Class I phosphosites')
    fig.update_xaxes(categoryorder='array', categoryarray=STATE_ORDER)
    fig.update_yaxes(range=[0, panel_max * 1.2])         # 10% headroom above max point
    fig.show()
    fig.write_image(rf'D:\Projects\nanoPhos\figures_upd\figure3\suppl_figure2d_{n}cells.pdf', width=500, height=500)

 100 cells | 2iL: mean=1168  SL: mean=752  RA: mean=945
 300 cells | 2iL: mean=2342  SL: mean=2531  RA: mean=3103
 800 cells | 2iL: mean=5887  SL: mean=6311  RA: mean=7410
1000 cells | 2iL: mean=6906  SL: mean=6702  RA: mean=8391
3000 cells | 2iL: mean=11749  SL: mean=11916  RA: mean=13519


In [9]:
# === PRIDE MetaInfo export (run after all panels above) ===
import sys; sys.path.insert(0, r"D:\Projects\nanoPhos_env\src")
from metainfo_export import dump_panel
from core import count_sites_per_sample_ptm_report, process_ptm_site_report
import numpy as np, pandas as pd
def _try(fn, sheet):
    try: fn()
    except Exception as e: print(f"  [SKIP {sheet}] {type(e).__name__}: {e}")

_try(lambda: dump_panel(pd.concat({f"{n}cells":cv_by_cells[n].reset_index(drop=True) for n in cv_by_cells},axis=1),
    "Suppl Figure 2a"),"Suppl Figure 2a")
_try(lambda: dump_panel(df_sel,"Suppl Figure 2b"),"Suppl Figure 2b")
def _s2c():
    rows=[]
    for n in counts_by_cells:
        for rep,v in enumerate(counts_by_cells[n],1):
            rows.append({"Condition":f"{n}cells","Replicate":rep,"Number of class I phosphosites":int(v)})
    dump_panel(pd.DataFrame(rows),"Suppl Figure 2c")
_try(_s2c,"Suppl Figure 2c")
def _s2d():
    rows=[]
    for n in depth_by_cells:
        for st,vals in depth_by_cells[n].items():
            for rep,v in enumerate(vals,1):
                rows.append({"Cells":n,"State":st,"Replicate":rep,"Number of class I phosphosites":int(v)})
    dump_panel(pd.DataFrame(rows),"Suppl Figure 2d")
_try(_s2d,"Suppl Figure 2d")
print("Suppl Figure 2 export done.")


  [MetaInfo] wrote 'Suppl Figure 2a'  (11014 rows x 6 cols)
  [MetaInfo] wrote 'Suppl Figure 2b'  (18 rows x 2 cols)
  [MetaInfo] wrote 'Suppl Figure 2c'  (18 rows x 3 cols)
  [MetaInfo] wrote 'Suppl Figure 2d'  (45 rows x 4 cols)
Suppl Figure 2 export done.
